# Chapter 17: Frequency–current curves — Brian2

Representative simulations adapted from [`python/chapter17.ipynb`](../python/chapter17.ipynb) and its [chapter notes](../python/chapter17.md).

Coverage: a numerical Wang–Buzsaki f–I experiment on a modest current grid, with a 500 ms transient removed and rates estimated from final interspike intervals. Independent neurons run in one Brian2 population. Source WB conductances and reversal potentials are retained; the source's long adaptive forward/backward continuation, onset magnifications, and other neuron models remain in Python. This independent-current scan does not claim to measure hysteresis.

WB uses instantaneous sodium activation with dynamic h and n. A steady periodic rate is $1000/\langle ISI\rangle$ Hz, with times in ms.

Run `uv sync --group dev` in the repository root and select that environment's Python kernel. These examples require the source checkout containing `mnd/brian`; an older PyPI installation does not contain these new helpers. Run all cells in order. Brian2 performs every trajectory integration. Numerical states use the book's units; helper clocks map a source time unit to one Brian millisecond. FHN and normal-form time is abstract model time.

In [ ]:
from pathlib import Path
import sys

# Use this source checkout so newly added helpers are available.
root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "mnd").is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))
import brian2 as b2
import numpy as np
import matplotlib.pyplot as plt
b2.prefs.codegen.target = "numpy"

In [ ]:
from mnd.brian.models import make_neurons

def simulate_wb_f_i(currents=np.linspace(0,1,11),duration=1000,dt=.02):
    currents=np.asarray(currents)
    neurons=make_neurons("wb",n=len(currents),current=currents,dt=dt)
    spikes=b2.SpikeMonitor(neurons)
    network=b2.Network(neurons,spikes)
    network.run(duration*b2.ms,namespace={})
    rates=np.zeros(len(currents))
    for i,times in spikes.spike_trains().items():
        times=np.asarray(times/b2.ms)
        tail=times[times>duration/2]
        if len(tail)>1:
            rates[i]=1000/np.mean(np.diff(tail[-5:]))
    return currents,rates

currents,rates=simulate_wb_f_i()
plt.figure(figsize=(7,4));plt.plot(currents,rates,"o-")
plt.xlabel("applied current (book units)");plt.ylabel("firing rate (Hz)")
plt.tight_layout();plt.show()
assert rates[0]==0 and rates[-1]>10
print("Rates (Hz):",np.round(rates,2))